# 🟡 L03 — Optional Extensions

> *Pure self-study material. Skipping this will not affect any later lesson. Come back when you want to deepen your understanding of the L03 concepts.*

This notebook covers four extensions to the core L03 content:

1. **Bias-variance tradeoff** — what it means and how to visualise it
2. **ROC curve + AUC** — an alternative way to summarise classifier performance
3. **Learning curves** — diagnose underfitting vs overfitting by training-set size
4. **Manual feature engineering** — adding derived features that improve the model

Each section is independent. Skip to whichever you're most curious about.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

# Re-load the NorthStar churn data and rebuild the pipeline
df = pd.read_csv("data/northstar_churn.csv")
y  = df["churned"]
X  = df.drop(columns=["customer_id", "churned"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)

numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

pipeline = Pipeline([("prep", preprocessor),
                     ("model", LogisticRegression(max_iter=1000, random_state=42))])
pipeline.fit(X_train, y_train)
print("✅ Setup complete.")

# 1. The Bias-Variance Tradeoff

**Bias** is the error from a model that's too simple to capture real patterns. A high-bias model gets training data wrong in the same way it gets test data wrong. *Symptom: training accuracy is poor.*

**Variance** is the error from a model that's too sensitive to specific training examples. A high-variance model nails the training data but flounders on new data. *Symptom: huge gap between training and test accuracy.*

**The tradeoff:** simpler models have higher bias and lower variance. More complex models have lower bias and higher variance. The sweet spot is where the SUM of bias and variance is minimised.

### Demonstration — increasing model complexity

We add polynomial interaction features to the logistic regression and watch training vs test accuracy diverge.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Just the numeric features for this demo
X_num = X[numeric_features].fillna(X[numeric_features].median())
X_num_train, X_num_test, y_num_train, y_num_test = train_test_split(
    X_num, y, test_size=0.20, stratify=y, random_state=42,
)

degrees = [1, 2, 3, 4, 5]
records = []

for deg in degrees:
    pipe = Pipeline([
        ("poly",   PolynomialFeatures(degree=deg, include_bias=False, interaction_only=False)),
        ("scl",    StandardScaler()),
        ("model",  LogisticRegression(max_iter=2000, C=1.0, random_state=42)),
    ])
    pipe.fit(X_num_train, y_num_train)
    train_score = pipe.score(X_num_train, y_num_train)
    test_score  = pipe.score(X_num_test,  y_num_test)
    n_features  = pipe.named_steps["poly"].n_output_features_
    records.append((deg, n_features, train_score, test_score))

bias_var_df = pd.DataFrame(records, columns=["degree", "n_features", "train_acc", "test_acc"])
bias_var_df["gap"] = bias_var_df["train_acc"] - bias_var_df["test_acc"]
print(bias_var_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(bias_var_df["degree"], bias_var_df["train_acc"], "o-", color="steelblue",
        linewidth=2, label="Training accuracy", markersize=8)
ax.plot(bias_var_df["degree"], bias_var_df["test_acc"], "s-", color="coral",
        linewidth=2, label="Test accuracy", markersize=8)
ax.fill_between(bias_var_df["degree"], bias_var_df["train_acc"], bias_var_df["test_acc"],
                color="lightgray", alpha=0.4, label="Gap (proxy for variance)")
ax.set_xlabel("Polynomial degree (model complexity →)")
ax.set_ylabel("Accuracy")
ax.set_title("Bias-Variance Tradeoff — accuracy vs model complexity")
ax.legend()
ax.set_xticks(degrees)
plt.tight_layout()
plt.show()

### 💡 What this tells us

- **Degree 1** is just standard logistic regression on the raw features. Training and test accuracy are nearly identical — the model is too simple to memorise specifics.
- **As degree grows**, training accuracy creeps up (or stays the same) but test accuracy can stagnate or fall. The gap between the two lines IS the variance.
- **The "best" degree** is where test accuracy peaks. Going beyond that is overfitting — you're memorising training-set noise.

**Rule of thumb:** start simple. Add complexity only when test performance genuinely improves. On this dataset, degree 1 is almost certainly enough.

# 2. ROC Curve and AUC

The **ROC curve** plots TRUE POSITIVE RATE (= recall) on the y-axis against FALSE POSITIVE RATE on the x-axis, as you sweep the threshold from 1 down to 0.

**AUC (Area Under the ROC Curve)** summarises that curve as a single number between 0 and 1:
- AUC = 0.5 → no better than random guessing
- AUC = 1.0 → perfect classifier
- AUC ≥ 0.7 → "okay" model; ≥ 0.8 → "good"; ≥ 0.9 → "very good"

**Why use AUC?** Unlike accuracy/precision/recall (which depend on a chosen threshold), AUC summarises the model's RANKING quality across ALL thresholds. Useful for comparing two models without committing to one threshold.

In [ ]:
y_proba = pipeline.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc                  = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, linewidth=2.5, color="steelblue", label=f"Logistic regression (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Random baseline (AUC = 0.5)")
ax.fill_between(fpr, tpr, alpha=0.15, color="steelblue")

# Mark a few specific thresholds
for t_target in [0.25, 0.5, 0.7]:
    i = np.argmin(np.abs(thresholds - t_target))
    ax.scatter([fpr[i]], [tpr[i]], s=100, color="indianred", zorder=5,
               edgecolor="white", linewidth=2)
    ax.annotate(f"T={t_target:.2f}", (fpr[i], tpr[i]), xytext=(10, 10),
                 textcoords="offset points", fontweight="bold", color="indianred")

ax.set_xlabel("False Positive Rate (1 − specificity)")
ax.set_ylabel("True Positive Rate (recall)")
ax.set_title(f"ROC Curve — NorthStar churn model\nAUC = {auc:.3f}")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"AUC: {auc:.3f}")
print()
print("Interpretation:")
if auc >= 0.85:   verdict = "very good"
elif auc >= 0.75: verdict = "good"
elif auc >= 0.65: verdict = "okay"
else:             verdict = "weak"
print(f"  → This is a '{verdict}' classifier in absolute terms.")
print(f"  → For comparison: AUC 0.5 = random; AUC 0.7-0.8 = useful for early warning;")
print(f"    AUC 0.9+ = very strong (often suggests too-good-to-be-true / data leakage on real data).")

### 💡 What this tells us

- **The ROC curve sweeps the threshold.** At T=0.7 (top-right area is most stringent), we have low FPR but also low TPR. At T=0.25 (bottom-left area), TPR is higher but so is FPR.
- **AUC summarises the curve.** It's the probability that a randomly chosen churner has a higher predicted probability than a randomly chosen stayer.
- **AUC ≠ accuracy.** AUC is INVARIANT to the class balance — much more useful than accuracy on imbalanced datasets like this one.

# 3. Learning Curves

A learning curve plots training and validation accuracy as you increase the training-set size. It diagnoses bias vs variance at a glance:

- **High bias (underfitting):** both curves plateau at LOW accuracy. Adding more data doesn't help.
- **High variance (overfitting):** training stays high, validation stays low. A big gap. More data helps.
- **Just right:** both curves converge toward each other at a high accuracy.

In [ ]:
train_sizes = np.linspace(0.1, 1.0, 8)

train_sizes_abs, train_scores, val_scores = learning_curve(
    pipeline,
    X_train, y_train,
    train_sizes=train_sizes,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std,
                alpha=0.2, color="steelblue")
ax.plot(train_sizes_abs, train_mean, "o-", linewidth=2, color="steelblue",
        label="Training accuracy")
ax.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std,
                alpha=0.2, color="coral")
ax.plot(train_sizes_abs, val_mean, "s-", linewidth=2, color="coral",
        label="Validation accuracy")
ax.set_xlabel("Training-set size")
ax.set_ylabel("Accuracy")
ax.set_title("Learning curve — NorthStar churn / logistic regression")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"Final training accuracy:   {train_mean[-1]:.3f}")
print(f"Final validation accuracy: {val_mean[-1]:.3f}")
print(f"Gap:                       {train_mean[-1] - val_mean[-1]:.3f}")

### 💡 What this tells us

- **Training and validation curves converge** — the gap is small. That's the signature of a *well-fit* model.
- **The asymptote is at a moderate accuracy** — both curves plateau. Adding more data won't dramatically improve this model. The bottleneck is the model class (logistic regression), not data quantity.
- **The implication:** if Sarah wants better performance, she needs a richer model (next week's L04: tree ensembles), not more data.

# 4. Manual Feature Engineering

Sometimes adding a few hand-crafted derived features improves the model — especially when there are obvious interactions or domain knowledge that the raw features don't capture.

### Idea: high-risk profile features

- `is_short_tenure_high_returns` — new customers AND high return rate → very likely to churn
- `low_engagement_score` — combined low logins + low reviews + low purchases
- `low_value_tier_long_tenure` — premium customers with long tenure → very unlikely to churn

In [ ]:
def add_engineered_features(df):
    out = df.copy()

    # Binary indicator: short tenure + high return rate
    out["risk_new_returner"] = (
        (out["tenure_months"] < 12) &
        (out["returns_per_purchase"] > 0.10)
    ).astype(int)

    # Composite low-engagement score (higher = less engaged)
    purchases_norm = out["num_purchases_quarter"].clip(0, 30) / 30.0
    last_login_norm = out["last_login_days_ago"].fillna(60).clip(0, 120) / 120.0
    out["engagement_score"] = 1.0 - purchases_norm + last_login_norm

    # Loyalty signal: premium + long tenure
    out["loyalty_signal"] = (
        (out["subscription_tier"] == "premium") &
        (out["tenure_months"] >= 24)
    ).astype(int)

    return out

X_train_eng = add_engineered_features(X_train)
X_test_eng  = add_engineered_features(X_test)

new_numeric = numeric_features + ["risk_new_returner", "engagement_score", "loyalty_signal"]

preprocessor_eng = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), new_numeric),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

pipeline_eng = Pipeline([("prep", preprocessor_eng),
                         ("model", LogisticRegression(max_iter=1000, random_state=42))])
pipeline_eng.fit(X_train_eng, y_train)

# Compare
y_proba_orig = pipeline.predict_proba(X_test)[:, 1]
y_proba_eng  = pipeline_eng.predict_proba(X_test_eng)[:, 1]

auc_orig = roc_auc_score(y_test, y_proba_orig)
auc_eng  = roc_auc_score(y_test, y_proba_eng)

print(f"AUC without engineered features: {auc_orig:.3f}")
print(f"AUC WITH engineered features:    {auc_eng:.3f}")
print(f"Lift:                            {auc_eng - auc_orig:+.3f}")

### 💡 What this tells us

- **Engineered features can boost AUC modestly.** The lift here is small because logistic regression can already model most interactions implicitly through its linear combination of standardised features.
- **For tree-based models (L04)**, the lift from these engineered features is often LARGER — because trees can't multiply features the way linear models can. Save these features for next week.
- **Domain knowledge beats algorithm choice.** A well-thought-out feature often improves performance more than swapping the algorithm. If Sarah talks to Aisha and learns "returns within the first 30 days are predictive of churn," that's worth more than another model.

---

## What you've covered

- **Bias-variance tradeoff** — a polynomial-degree sweep made the tradeoff visible
- **ROC curve & AUC** — threshold-independent summary of classifier quality
- **Learning curves** — diagnostic for whether you need more data or a different model
- **Manual feature engineering** — combine raw features to encode domain knowledge

**Back to the main flow:** none of this is required for the assignment or for L04. It's here when you want a deeper view of WHY the L03 pipeline behaves the way it does.